# 04 — Achsen-Ablation: 3 gegen 6 Achsen

Entspricht Kapitel 2 (*Sensorempfehlung*) und deren empirischer
Pruefung in Kapitel 5.

Die Empfehlung zugunsten eines 6-Achsen-Sensors stuetzte sich bisher
auf die Feature-Importance des Random Forest — ein indirekter Beleg.
Hier wird direkt gemessen, wie viel Sensitivitaet der Verzicht auf
das Gyroskop kostet.

Drei Modelle, identische Architektur, identische Aufteilung,
identische Hyperparameter. Einziger Unterschied: die Eingangskanaele.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q scipy scikit-learn tensorflow openpyxl

import os, sys, shutil, time, json
from pathlib import Path
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Gemeinsame Bibliothek holen (einzige Quelle fuer alle Loader)
# fall_lib laden: zuerst GitHub, sonst Kopie auf Drive. Schlaegt beides
# fehl, wird abgebrochen -- ohne fall_lib waeren alle Ergebnisse falsch
# (andere Fensterlaenge, anderes Heruntertasten).
import urllib.request, shutil as _sh
_LIB_URL = "https://raw.githubusercontent.com/Alyelb/ML-Modell-for-Fall-detection/main/fall_lib.py"
_LIB_DRIVE = "/content/drive/MyDrive/ML_Fall_Detection/fall_lib.py"
try:
    urllib.request.urlretrieve(_LIB_URL, "/content/fall_lib.py")
    print("[fall_lib] von GitHub geladen")
except Exception as _e:
    print(f"[fall_lib] GitHub nicht erreichbar ({_e}) -- nutze Drive-Kopie")
    _sh.copy(_LIB_DRIVE, "/content/fall_lib.py")
sys.path.insert(0, "/content")
import fall_lib as fl
import importlib; importlib.reload(fl)
# Pflichtfunktionen pruefen
for _f in ("new_run", "build_windows", "znorm", "WIN_LEN"):
    assert hasattr(fl, _f), f"fall_lib unvollstaendig ({_f} fehlt) -- Datei pruefen."
print("[fall_lib] Fensterlaenge", fl.WIN_LEN, "bei", fl.FS_TARGET, "Hz")

BASE  = "/content/drive/MyDrive/ML_Fall_Detection"
fl.RESULTS_ROOT = f"{BASE}/results"
MODEL = f"{BASE}/results/fall_model.keras"      # einheitlicher Modellpfad

print("Konfiguration:", fl.default_config())

Mounted at /content/drive
Konfiguration: {'fs_target': 50, 'window_sec': 2.0, 'overlap': 0.5, 'window_len': 100, 'step': 50, 'kfall_fall_tasks': 'T20-T34'}


In [2]:
def hole(name, ziel):
    """Kopiert einen Datensatz von Drive auf die lokale Platte.
    Drive ist fuer tausende Kleindateien zu langsam."""
    src = f"{BASE}/{name}"
    if Path(ziel).exists():
        print(f"[ok] {name} bereits lokal"); return ziel
    if not Path(src).exists():
        print(f"[!] {src} nicht gefunden. Vorhanden:")
        for p in sorted(Path(BASE).iterdir()):
            print("   ", p.name)
        return None
    t = time.time(); shutil.copytree(src, ziel)
    print(f"[ok] {name} kopiert in {time.time()-t:.0f}s")
    return ziel

In [3]:
run = fl.new_run("04_achsen")
ergebnisse = {}

# Position waehlen: SisFall (Huefte) oder UMAFall ANKLE (Knoechel)
# Der Knoechel ist fuer die Insole aussagekraeftiger, da dort die
# Beschleunigung durch Fersenauftritte gesaettigt ist.
DATENSATZ = "sisfall"      # "sisfall" oder "umafall_ankle"

if DATENSATZ == "sisfall":
    trials = fl.load_sisfall(hole("SisFall_dataset", "/content/SisFall"))
else:
    trials = fl.load_umafall(
        hole("UMAFall Fall Detection/UMAFall_Dataset_corrected",
             "/content/UMAFall"), position="ANKLE")

X, y, g, _ = fl.build_windows(trials, mode="standard")
tr, te = fl.subject_split(g, y, test_size=0.25)
print(f"\nTrain {len(tr)} | Test {len(te)} Fenster")

[Lauf] /content/drive/MyDrive/ML_Fall_Detection/results/04_achsen_2026-08-26_1107
[ok] SisFall_dataset kopiert in 1070s
[SisFall] 4504 Trials (1797 Sturz, 2707 ADL), 38 Probanden

Train 38974 | Test 12431 Fenster


In [4]:
varianten = {"acc6": "6 Achsen (Acc+Gyr)",
             "acc3": "3 Achsen (nur Acc)",
             "gyr3": "3 Achsen (nur Gyr)"}
modelle, abl = {}, {}

for mode, name in varianten.items():
    Xn = fl.znorm(fl.select_channels(X, mode))
    m = fl.build_cnn(n_channels=Xn.shape[2])
    fl.train(m, Xn[tr], y[tr], epochs=30)
    print(f"\n{name}  ({m.count_params():,} Parameter)")
    r = fl.evaluate(m, Xn[te], y[te], "  ")
    r["params"] = int(m.count_params())
    abl[mode] = r; modelle[mode] = m


6 Achsen (Acc+Gyr)  (61,601 Parameter)
                              Se=0.987  Sp=0.999  Acc=0.998  FP=   17  FN=   4

3 Achsen (nur Acc)  (61,121 Parameter)
                              Se=0.980  Sp=0.999  Acc=0.998  FP=   14  FN=   6

3 Achsen (nur Gyr)  (61,121 Parameter)
                              Se=0.930  Sp=0.995  Acc=0.993  FP=   64  FN=  21


## Ergebnis

In [5]:
print(f"{'Variante':<24} {'Se':>7} {'Sp':>7} {'Acc':>7} {'Param':>9}")
print("-"*58)
for k, name in varianten.items():
    r = abl[k]
    print(f"{name:<24} {r['Se']:>7.3f} {r['Sp']:>7.3f} "
          f"{r['Acc']:>7.3f} {r['params']:>9,}")

d = abl["acc6"]["Se"] - abl["acc3"]["Se"]
print(f"\nBeitrag des Gyroskops: {d*100:+.2f} Prozentpunkte "
      f"Sensitivitaet")
print(f"Differenz der Parameterzahl: "
      f"{abl['acc6']['params']-abl['acc3']['params']} "
      f"(nur die Eingangsschicht)")

if abl["acc6"]["Se"] > 0.97:
    print("\nHinweis: Beide Varianten liegen nahe der Obergrenze. "
          "Ein Deckeneffekt kann kleine Unterschiede verdecken; "
          "die Aussage gilt zunaechst nur fuer diese Position.")
ergebnisse["ablation"] = abl
ergebnisse["delta_se"] = round(float(d), 4)
ergebnisse["datensatz"] = DATENSATZ

Variante                      Se      Sp     Acc     Param
----------------------------------------------------------
6 Achsen (Acc+Gyr)         0.987   0.999   0.998    61,601
3 Achsen (nur Acc)         0.980   0.999   0.998    61,121
3 Achsen (nur Gyr)         0.930   0.995   0.993    61,121

Beitrag des Gyroskops: +0.67 Prozentpunkte Sensitivitaet
Differenz der Parameterzahl: 480 (nur die Eingangsschicht)

Hinweis: Beide Varianten liegen nahe der Obergrenze. Ein Deckeneffekt kann kleine Unterschiede verdecken; die Aussage gilt zunaechst nur fuer diese Position.


## Externe Gegenprobe: UniMiB-SHAR

UniMiB-SHAR enthaelt ausschliesslich Beschleunigungsdaten und eignet
sich daher zur unabhaengigen Pruefung des 3-Achsen-Modells.

Zu beachten: Die Daten stammen von einem Smartphone mit einem
Messbereich von nur ±2 g; Sturzspitzen werden abgeschnitten.

In [6]:
UNI = hole("UniMiB-SHAR/data", "/content/UniMiB")
if UNI:
    Xu, yu, gu = fl.load_unimib(UNI)
    Xu = fl.znorm(Xu)
    print()
    zs = fl.evaluate(modelle["acc3"], Xu, yu, "Zero-Shot")
    tru, teu = fl.subject_split(gu, yu, test_size=0.3)
    m_ft = fl.finetune(modelle["acc3"], Xu[tru], yu[tru])
    ft = fl.evaluate(m_ft, Xu[teu], yu[teu], "Fine-Tuned")
    ergebnisse["unimib"] = {"zero_shot": zs, "fine_tuned": ft}

fl.save_results(run, ergebnisse, config=fl.default_config())

[ok] UniMiB-SHAR/data kopiert in 108s
[UniMiB] 11771 Fenster (4192 Sturz, 7579 ADL), 30 Probanden, nur 3 Achsen
[UniMiB] Messbereich des Smartphones: max 2.00 g

  Zero-Shot                   Se=0.204  Sp=0.995  Acc=0.713  FP=   40  FN=3337
  Fine-Tuned                  Se=0.959  Sp=0.976  Acc=0.970  FP=   55  FN=  56
[gesichert] /content/drive/MyDrive/ML_Fall_Detection/results/04_achsen_2026-08-26_1107/results.json


PosixPath('/content/drive/MyDrive/ML_Fall_Detection/results/04_achsen_2026-08-26_1107/results.json')